# Sub-objective 2 — Modeling with Transfer Learning
**Translator: English → Ekegusii AND Kiswahili → Ekegusii**

Run in **Google Colab** with GPU: `Runtime > Change runtime type > T4 GPU`.

This notebook fulfils, in order:
1. Experiment tracking (Weights & Biases)
2. ≥2 pretrained models (mT5-small, NLLB-200-distilled-600M), few-shot fine-tuned
3. Low-resource handling: layer freezing + multi-source data combination (documented as augmentation)
4. Ablations: zero-shot vs few-shot, domain adaptation
5. Checkpoints + logs saved to Drive
6. Hyperparameters / training time / results documented automatically
7. Mid-week check-in notes (GPU/Colab troubleshooting)
8. Inference script + performance summary (Week 3 deliverable)

**Design note:** English and Kiswahili are both fine-tuned as sources → Ekegusii as the single
target. This is a genuine data-combination technique for the low-resource side (Ekegusii is
what's actually scarce), not two unrelated tasks.


## 1. Setup

In [ ]:
!nvidia-smi

In [1]:
!pip install -q transformers datasets accelerate sentencepiece sacrebleu evaluate wandb


In [2]:
import wandb
wandb.login()  # paste your API key from wandb.ai/authorize when prompted


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: 21ibtj (21ibtj-usiu) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [3]:
import os, json, time, random
import numpy as np
import pandas as pd
import torch

from transformers import set_seed

# Reproducibility
set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

GPU available: True
Device: Tesla T4


## 2. Load curated dataset

In [5]:
# from google.colab import files
# uploaded = files.upload()  # choose Final_merged_psas.csv
# csv_path = list(uploaded.keys())[0]


# Kaggle: after adding your CSV via "Add Input" -> Upload, it lands under /kaggle/input/<dataset-name>/
# Update the dataset folder name below to match what Kaggle assigned when you uploaded it.
csv_path = "/kaggle/input/datasets/hannahailemariam/final-merged-psas/Final_merged_psas.csv"

In [6]:
df = pd.read_csv(csv_path, encoding="utf-8-sig")
df.columns = [c.strip() for c in df.columns]
print(df.shape)
df.head()


(21306, 5)


,PSA_ID,Domain,English,Kiswahili,Ekegusii
0,1,Agriculture,Farmers are urged to prioritize safe agrochemi...,Wakulima wanakumbushwa kuzingatia matumizi sal...,Abakuli batosere omogori bw'ogenda gesia bw'og...
1,2,Agriculture,Trucks ferrying top-dressing fertilizer are no...,Masafa yanayosafirisha mbolea ya kuongeza mavu...,Matika agwanana oria okora buya bwakonyang'ana...
2,3,Agriculture,Farmers in Wajir are invited to participate in...,Wakulima wa Wajir wanakaribishwa kushiriki kat...,Abagere bu Wajir batarikire kugana mu Ksh. 5 b...
3,4,Agriculture,Farmers are encouraged to participate in the s...,Wakulima wanahimizwa kushiriki katika mpango w...,Abagaba batemerewe kuhakanya mulashi wa ethano...
4,5,Agriculture,A Ksh. 34.4 billion program has been launched ...,Mpango wa Ksh. bilioni 34.4 umeanzishwa ili ku...,Programu ya Ksh. 34.4 bilioni imeanzishwa kuim...


### 2.1 Build the combined (English + Kiswahili) → Ekegusii dataset

Each row becomes **two** training examples where possible: one with English as source,
one with Kiswahili as source, both targeting the same Ekegusii text. Rows missing
Kiswahili only contribute the English→Ekegusii example.

Combining both source languages into one training set is our **data augmentation**
technique for the low-resource target side (Ekegusii): the decoder sees roughly twice
as many Ekegusii target examples as either source alone would give it.


In [7]:
def build_combined(df):
    rows = []
    for _, r in df.iterrows():
        if pd.notna(r["English"]) and str(r["English"]).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["English"], "target_text": r["Ekegusii"],
                "source_lang": "en"
            })
        if pd.notna(r.get("Kiswahili")) and str(r.get("Kiswahili")).strip():
            rows.append({
                "PSA_ID": r["PSA_ID"], "Domain": r["Domain"],
                "source_text": r["Kiswahili"], "target_text": r["Ekegusii"],
                "source_lang": "sw"
            })
    return pd.DataFrame(rows).dropna(subset=["target_text"])

combined = build_combined(df)
combined = combined[combined["target_text"].astype(str).str.strip() != ""]
print("Total combined examples:", len(combined))
print(combined["source_lang"].value_counts())
print(combined["Domain"].value_counts())


Total combined examples: 42609
source_lang
en    21306
sw    21303
Name: count, dtype: int64
Domain
Education            10573
Agriculture           8912
Health                8176
Security & Safety     8028
Governance            6920
Name: count, dtype: int64


In [8]:
from sklearn.model_selection import train_test_split

# stratify by source_lang so both directions are represented in every split
train_df, temp_df = train_test_split(combined, test_size=0.2, random_state=42,
                                      stratify=combined["source_lang"])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42,
                                    stratify=temp_df["source_lang"])

print("Train:", len(train_df), " Val:", len(val_df), " Test:", len(test_df))
train_df["source_lang"].value_counts(), test_df["source_lang"].value_counts()


Train: 34087  Val: 4261  Test: 4261


(source_lang
 en    17045
 sw    17042
 Name: count, dtype: int64,
 source_lang
 en    2131
 sw    2130
 Name: count, dtype: int64)

**Low-resource note:** Ekegusii is not in mT5's or NLLB-200's pretraining language list,
so this is a genuine low-resource target. English is well covered by both models; Kiswahili
is well covered by both models. The combination strategy above specifically compensates for
the scarce side (Ekegusii target data), not the source side.


## 3. Shared utilities: tokenization, metrics, layer freezing, timing

In [9]:
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import evaluate

sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def to_hf(d):
    return Dataset.from_pandas(d[["source_text", "target_text", "source_lang", "Domain"]]
                                .reset_index(drop=True))

train_ds = to_hf(train_df)
val_ds   = to_hf(val_df)
test_ds  = to_hf(test_df)

def build_compute_metrics(tokenizer):
    def compute_metrics(eval_preds):
        preds, labels = eval_preds
        if isinstance(preds, tuple):
            preds = preds[0]
        preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
        decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
        labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
        decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
        bleu = sacrebleu.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        c = chrf.compute(predictions=decoded_preds, references=[[l] for l in decoded_labels])
        return {"bleu": bleu["score"], "chrf": c["score"]}
    return compute_metrics

def freeze_encoder_layers(model, num_layers_to_freeze):
    """Freeze bottom N encoder layers to reduce overfitting risk on our small,
    low-resource fine-tuning set and cut compute cost."""
    encoder = model.get_encoder()
    layers = encoder.block if hasattr(encoder, "block") else encoder.layers
    for i, layer in enumerate(layers):
        if i < num_layers_to_freeze:
            for p in layer.parameters():
                p.requires_grad = False
    return model

def score(preds, refs, label, log_to_wandb=True):
    bleu = sacrebleu.compute(predictions=preds, references=[[r] for r in refs])
    c = chrf.compute(predictions=preds, references=[[r] for r in refs])
    print(f"{label:35s} BLEU={bleu['score']:.2f}  chrF={c['score']:.2f}")
    if log_to_wandb and wandb.run is not None:
        wandb.log({f"{label}_bleu": bleu["score"], f"{label}_chrf": c["score"]})
    return {"label": label, "bleu": bleu["score"], "chrf": c["score"]}

MAX_LEN = 128
results_log = []          # collects every score() call for the final summary table
timing_log = {}           # collects wall-clock training time per model


## 4. Model A — mT5-small

Uses a text prefix to indicate source language, since mT5 has no dedicated language tokens.


In [10]:
MT5_CHECKPOINT = "google/mt5-small"
mt5_tok = AutoTokenizer.from_pretrained(MT5_CHECKPOINT)

def mt5_prefix(source_lang):
    return "translate English to Ekegusii: " if source_lang == "en" else "translate Kiswahili to Ekegusii: "

def preprocess_mt5(batch):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(batch["source_lang"], batch["source_text"])]
    model_inputs = mt5_tok(inputs, max_length=MAX_LEN, truncation=True)
    labels = mt5_tok(text_target=batch["target_text"], max_length=MAX_LEN, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok_mt5 = train_ds.map(preprocess_mt5, batched=True)
val_tok_mt5   = val_ds.map(preprocess_mt5, batched=True)


Map:   0%|          | 0/34087 [00:00<?, ? examples/s]

Map:   0%|          | 0/4261 [00:00<?, ? examples/s]

### 4.1 Baseline (zero-shot) — mT5, before any fine-tuning

In [11]:
base_mt5 = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
base_mt5.to("cuda" if torch.cuda.is_available() else "cpu")

def generate_mt5(model, texts, source_langs, max_new_tokens=MAX_LEN, batch_size=16):
    inputs = [mt5_prefix(sl) + t for sl, t in zip(source_langs, texts)]
    all_preds = []
    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i + batch_size]
        enc = mt5_tok(batch, return_tensors="pt", padding=True, truncation=True,
                      max_length=MAX_LEN).to(model.device)
        out = model.generate(**enc, max_length=max_new_tokens)
        all_preds.extend(mt5_tok.batch_decode(out, skip_special_tokens=True))
        del enc, out
        torch.cuda.empty_cache()
    return all_preds

# Evaluate baseline separately for each source language (needed for the domain/ablation tables)
test_en = test_df[test_df["source_lang"] == "en"]
test_sw = test_df[test_df["source_lang"] == "sw"]

preds_base_en = generate_mt5(base_mt5, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_base_en, list(test_en["target_text"]), "mt5_zero-shot_en-guz"))

preds_base_sw = generate_mt5(base_mt5, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_base_sw, list(test_sw["target_text"]), "mt5_zero-shot_sw-guz"))

del base_mt5
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


mt5_zero-shot_en-guz                BLEU=0.01  chrF=1.18
mt5_zero-shot_sw-guz                BLEU=0.01  chrF=1.45


### 4.2 Fine-tuning — mT5 (few-shot), with layer freezing

In [12]:
wandb.init(project="psa-translation", name="mt5-small_combined-guz", reinit=True)

mt5_model = AutoModelForSeq2SeqLM.from_pretrained(MT5_CHECKPOINT)
mt5_model = freeze_encoder_layers(mt5_model, num_layers_to_freeze=4)  # mt5-small: 8 encoder layers

data_collator_mt5 = DataCollatorForSeq2Seq(mt5_tok, model=mt5_model)

args_mt5 = Seq2SeqTrainingArguments(
    output_dir="checkpoints/mt5_combined_guz",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    num_train_epochs=5,
    save_total_limit=2,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    report_to="wandb",
    run_name="mt5-small_combined-guz",
    fp16=False,  # mT5 unstable in fp16 on T4 -- train in fp32
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
    gradient_checkpointing=True,
)

trainer_mt5 = Seq2SeqTrainer(
    model=mt5_model,
    args=args_mt5,
    train_dataset=train_tok_mt5,
    eval_dataset=val_tok_mt5,
    data_collator=data_collator_mt5,
    processing_class=mt5_tok,
    compute_metrics=build_compute_metrics(mt5_tok),
)

t0 = time.time()
trainer_mt5.train(resume_from_checkpoint="checkpoints/mt5_combined_guz/checkpoint-6393")
timing_log["mt5_train_seconds"] = time.time() - t0
print(f"mT5 training time: {timing_log['mt5_train_seconds']/60:.1f} minutes")

trainer_mt5.save_model("checkpoints/mt5_combined_guz/final")
wandb.log({"mt5_train_seconds": timing_log["mt5_train_seconds"]})


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return su

Epoch,Training Loss,Validation Loss,Bleu,Chrf
4,6.777495,3.166998,3.302885,26.161544
5,6.705437,3.145593,3.634703,26.930447


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

mT5 training time: 102.5 minutes


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

### 4.3 Fine-tuned evaluation — mT5, per source language

In [13]:
preds_ft_en = generate_mt5(trainer_mt5.model, list(test_en["source_text"]), list(test_en["source_lang"]))
results_log.append(score(preds_ft_en, list(test_en["target_text"]), "mt5_few-shot_en-guz"))

preds_ft_sw = generate_mt5(trainer_mt5.model, list(test_sw["source_text"]), list(test_sw["source_lang"]))
results_log.append(score(preds_ft_sw, list(test_sw["target_text"]), "mt5_few-shot_sw-guz"))


mt5_few-shot_en-guz                 BLEU=3.98  chrF=26.84
mt5_few-shot_sw-guz                 BLEU=3.41  chrF=26.11


In [19]:
import json
with open("checkpoints/mt5_combined_guz/final_test_results.json", "w") as f:
    json.dump({
        "mt5_few-shot_en-guz": {"bleu": 3.98, "chrf": 26.84},
        "mt5_few-shot_sw-guz": {"bleu": 3.41, "chrf": 26.11},
    }, f, indent=2)


In [17]:
import os
print(os.path.exists("checkpoints/mt5_combined_guz/final_test_results.json"))

True


In [20]:
import os
print(os.listdir("checkpoints/mt5_combined_guz"))

['checkpoint-10655', 'checkpoint-6393', 'final_test_results.json', 'final']


In [21]:
from IPython.display import FileLink
FileLink("checkpoints/mt5_combined_guz/final_test_results.json")

/kaggle/working/checkpoints/mt5_combined_guz/final_test_results.json

In [22]:
import shutil
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"Used: {used/1e9:.2f} GB, Free: {free/1e9:.2f} GB")

Used: 15.48 GB, Free: 5.46 GB


In [23]:
import shutil
from IPython.display import FileLink

shutil.make_archive("mt5_final_backup", "zip", "checkpoints/mt5_combined_guz/final")
FileLink("mt5_final_backup.zip")

/kaggle/working/mt5_final_backup.zip

## 5. Model B — NLLB-200-distilled-600M

Uses native language-code tokens for the sources (`eng_Latn`, `swh_Latn`). Ekegusii has no
native NLLB code, so we use the closest supported code as a placeholder decoder tag — a
deliberate, documented choice, not an oversight.


In [ ]:
NLLB_CHECKPOINT = "facebook/nllb-200-distilled-600M"
TGT_PLACEHOLDER = "swh_Latn"  # placeholder tag for Ekegusii (unsupported by NLLB-200)

nllb_tok = AutoTokenizer.from_pretrained(NLLB_CHECKPOINT)

def nllb_src_code(source_lang):
    return "eng_Latn" if source_lang == "en" else "swh_Latn"

def preprocess_nllb(batch):
    # group by source lang within the batch since src_lang is a tokenizer-level setting
    all_ids, all_labels = [], []
    for sl, src, tgt in zip(batch["source_lang"], batch["source_text"], batch["target_text"]):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(src, max_length=MAX_LEN, truncation=True)
        with nllb_tok.as_target_tokenizer():
            lab = nllb_tok(tgt, max_length=MAX_LEN, truncation=True)
        all_ids.append(enc)
        all_labels.append(lab["input_ids"])
    return {
        "input_ids": [e["input_ids"] for e in all_ids],
        "attention_mask": [e["attention_mask"] for e in all_ids],
        "labels": all_labels,
    }

train_tok_nllb = train_ds.map(preprocess_nllb, batched=True, batch_size=16)
val_tok_nllb   = val_ds.map(preprocess_nllb, batched=True, batch_size=16)


### 5.1 Baseline (zero-shot) — NLLB, before any fine-tuning

In [ ]:
base_nllb = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
base_nllb.to("cuda" if torch.cuda.is_available() else "cpu")

def generate_nllb(model, texts, source_langs, tgt_code=TGT_PLACEHOLDER):
    preds = []
    for sl, t in zip(source_langs, texts):
        nllb_tok.src_lang = nllb_src_code(sl)
        enc = nllb_tok(t, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = nllb_tok.convert_tokens_to_ids(tgt_code)
        out = model.generate(**enc, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        preds.append(nllb_tok.decode(out[0], skip_special_tokens=True))
    return preds

# NLLB inference is per-example (language-tagged), so we subsample the test set for speed
# on Colab's free tier -- adjust n_eval upward if you have GPU time to spare
n_eval = min(60, len(test_en))
preds_base_en_nllb = generate_nllb(base_nllb, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_base_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_zero-shot_en-guz"))

n_eval_sw = min(60, len(test_sw))
preds_base_sw_nllb = generate_nllb(base_nllb, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_base_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_zero-shot_sw-guz"))

del base_nllb
torch.cuda.empty_cache()


### 5.2 Fine-tuning — NLLB (few-shot), with layer freezing

In [ ]:
wandb.init(project="psa-translation", name="nllb-600M_combined-guz", reinit=True)

nllb_model = AutoModelForSeq2SeqLM.from_pretrained(NLLB_CHECKPOINT)
nllb_model = freeze_encoder_layers(nllb_model, num_layers_to_freeze=6)  # heavier model -> freeze more

args_nllb = Seq2SeqTrainingArguments(
    output_dir="checkpoints/nllb_combined_guz",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=1e-4,
    num_train_epochs=5,
    predict_with_generate=True,
    generation_max_length=MAX_LEN,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=20,
    report_to="wandb",
    run_name="nllb-600M_combined-guz",
    fp16=torch.cuda.is_available(),
    load_best_model_at_end=True,
    metric_for_best_model="bleu",
)

trainer_nllb = Seq2SeqTrainer(
    model=nllb_model,
    args=args_nllb,
    train_dataset=train_tok_nllb,
    eval_dataset=val_tok_nllb,
    data_collator=DataCollatorForSeq2Seq(nllb_tok, model=nllb_model),
    processing_class=nllb_tok,
    compute_metrics=build_compute_metrics(nllb_tok),
)

t0 = time.time()
trainer_nllb.train()
timing_log["nllb_train_seconds"] = time.time() - t0
print(f"NLLB training time: {timing_log['nllb_train_seconds']/60:.1f} minutes")

trainer_nllb.save_model("checkpoints/nllb_combined_guz/final")
wandb.log({"nllb_train_seconds": timing_log["nllb_train_seconds"]})


In [ ]:
import os
for root, dirs, files in os.walk("checkpoints/mt5_combined_guz"):
    print(root, "-", len(files), "files")

### 5.3 Fine-tuned evaluation — NLLB, per source language

In [ ]:
preds_ft_en_nllb = generate_nllb(trainer_nllb.model, list(test_en["source_text"])[:n_eval], list(test_en["source_lang"])[:n_eval])
results_log.append(score(preds_ft_en_nllb, list(test_en["target_text"])[:n_eval], "nllb_few-shot_en-guz"))

preds_ft_sw_nllb = generate_nllb(trainer_nllb.model, list(test_sw["source_text"])[:n_eval_sw], list(test_sw["source_lang"])[:n_eval_sw])
results_log.append(score(preds_ft_sw_nllb, list(test_sw["target_text"])[:n_eval_sw], "nllb_few-shot_sw-guz"))


## 6. Ablation study 2 — Domain adaptation

Breaks down the **fine-tuned mT5** model's performance by PSA domain (Agriculture, Health,
Education, Governance, Security), to see whether some domains translate better than others.


In [ ]:
domain_results = []
for dom in test_df["Domain"].unique():
    sub = test_df[test_df["Domain"] == dom]
    if len(sub) < 5:
        continue  # skip domains with too few test examples to score meaningfully
    preds = generate_mt5(trainer_mt5.model, list(sub["source_text"]), list(sub["source_lang"]))
    r = score(preds, list(sub["target_text"]), f"mt5_few-shot_domain-{dom}")
    r["domain"] = dom
    r["n_examples"] = len(sub)
    domain_results.append(r)

domain_df = pd.DataFrame(domain_results)
domain_df


## 7. Save checkpoints and logs to Google Drive

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

# import shutil
# shutil.copytree("checkpoints", "/content/drive/MyDrive/psa_checkpoints", dirs_exist_ok=True)

# # also persist the results tables as CSV so they survive the session
# results_df = pd.DataFrame(results_log)
# results_df.to_csv("/content/drive/MyDrive/psa_checkpoints/ablation_results.csv", index=False)
# domain_df.to_csv("/content/drive/MyDrive/psa_checkpoints/domain_results.csv", index=False)
# with open("/content/drive/MyDrive/psa_checkpoints/timing_log.json", "w") as f:
#     json.dump(timing_log, f, indent=2)

# print("Checkpoints and logs saved to Google Drive.")


In [ ]:
import shutil, os

os.makedirs("/kaggle/working/psa_checkpoints", exist_ok=True)
shutil.copytree("checkpoints", "/kaggle/working/psa_checkpoints", dirs_exist_ok=True)

results_df = pd.DataFrame(results_log)
results_df.to_csv("/kaggle/working/psa_checkpoints/ablation_results.csv", index=False)
domain_df.to_csv("/kaggle/working/psa_checkpoints/domain_results.csv", index=False)
with open("/kaggle/working/psa_checkpoints/timing_log.json", "w") as f:
    json.dump(timing_log, f, indent=2)

print("Checkpoints and logs saved to /kaggle/working — they'll appear as notebook Output after you save/commit.")

## 8. Mid-week check-in — GPU/Colab troubleshooting notes

Pause here partway through the week and check:

- **"CUDA out of memory"** → lower `per_device_train_batch_size` (try 2) and raise
  `gradient_accumulation_steps` to compensate; restart runtime to clear GPU memory first.
- **Session disconnected / runtime reset** → re-run Section 1 (installs + login), then
  reload checkpoints from Drive with `AutoModelForSeq2SeqLM.from_pretrained("/content/drive/MyDrive/psa_checkpoints/<model>/final")`
  instead of retraining from scratch.
- **Free GPU quota running low** → prioritize finishing mT5 (both directions) since it's
  much cheaper to train than NLLB-600M; NLLB can run last or be reduced in `num_train_epochs`.
- **W&B not logging** → confirm `wandb.init()` ran without error and `report_to="wandb"`
  is set in the `Seq2SeqTrainingArguments` before training starts.

Record here what you actually hit this week, e.g.:
> *"Hit OOM on NLLB at batch size 8 — reduced to 4 + grad accumulation 2, resolved. Colab
> disconnected once after ~90 min idle; resumed from Drive checkpoint without retraining mT5."*


## 9. Inference script (Week 3 deliverable)

In [ ]:
def translate_psa(text, source_lang="en", model_choice="mt5"):
    """
    text: input sentence
    source_lang: 'en' (English) or 'sw' (Kiswahili)
    model_choice: 'mt5' or 'nllb'
    Returns: Ekegusii translation
    """
    if model_choice == "mt5":
        model = trainer_mt5.model
        inputs = mt5_tok(mt5_prefix(source_lang) + text, return_tensors="pt",
                          truncation=True, max_length=MAX_LEN).to(model.device)
        out = model.generate(**inputs, max_length=MAX_LEN)
        return mt5_tok.decode(out[0], skip_special_tokens=True)
    else:
        model = trainer_nllb.model
        nllb_tok.src_lang = nllb_src_code(source_lang)
        inputs = nllb_tok(text, return_tensors="pt", truncation=True, max_length=MAX_LEN).to(model.device)
        forced_bos = nllb_tok.convert_tokens_to_ids(TGT_PLACEHOLDER)
        out = model.generate(**inputs, forced_bos_token_id=forced_bos, max_length=MAX_LEN)
        return nllb_tok.decode(out[0], skip_special_tokens=True)

# --- Demo: sample PSAs ---
samples = [
    ("Farmers are urged to prioritize safe agrochemical usage this season.", "en"),
    ("Wakulima wanahimizwa kutumia kemikali za kilimo kwa usalama msimu huu.", "sw"),
]

for text, lang in samples:
    print(f"[{lang}] {text}")
    print("  mT5  ->", translate_psa(text, source_lang=lang, model_choice="mt5"))
    print("  NLLB ->", translate_psa(text, source_lang=lang, model_choice="nllb"))
    print()


## 10. Hyperparameters, training time, and preliminary results

The two cells below auto-generate this documentation from what actually ran — fill in
the narrative sentences after reading the printed tables.


In [ ]:
hyperparam_table = pd.DataFrame([
    {"Model": "mT5-small", "Pair": "combined (en+sw)->guz", "Epochs": args_mt5.num_train_epochs,
     "Batch size": args_mt5.per_device_train_batch_size, "LR": args_mt5.learning_rate,
     "Frozen layers": "4/8", "Train time (min)": round(timing_log.get("mt5_train_seconds", 0)/60, 1)},
    {"Model": "NLLB-200-distilled-600M", "Pair": "combined (en+sw)->guz", "Epochs": args_nllb.num_train_epochs,
     "Batch size": f"{args_nllb.per_device_train_batch_size} (x{args_nllb.gradient_accumulation_steps} accum)",
     "LR": args_nllb.learning_rate, "Frozen layers": "6/12",
     "Train time (min)": round(timing_log.get("nllb_train_seconds", 0)/60, 1)},
])
hyperparam_table


In [ ]:
print("=== Zero-shot vs Few-shot ablation ===")
print(results_df.to_string(index=False))
print()
print("=== Domain adaptation ablation (mT5, few-shot) ===")
print(domain_df[["domain", "n_examples", "bleu", "chrf"]].to_string(index=False))


**Preliminary observations (fill in with your actual numbers above):**
- English→Ekegusii vs Kiswahili→Ekegusii: ...
- Zero-shot vs few-shot gap (evidence fine-tuning helped): ...
- Best/worst performing domain and a possible reason why: ...
- mT5 vs NLLB comparison, given training time tradeoff: ...
- Mid-week issues encountered and how they were resolved: (see Section 8)
